# 01b · Baseline: silma-tts-derja (F5-TTS fine-tune)

Someone already fine-tuned F5-TTS on LinTO for Tunisian Derja. We run it with **your** voice as reference and the same lines as 01a.

- **Licence: CC BY-NC 4.0.** For comparison only. The release is built on Chatterbox (MIT).
- Use a **fresh runtime**, not the one from 01a: the two libraries pin different torch versions.
- The model author ships an English reference clip and says a native Derja clip would likely improve prosody. Your clip is that test.

In [ ]:
!pip install -q f5-tts==1.1.7 num2words

In [ ]:
REPO_URL = "https://github.com/YoussefKossentini/derja-tts"   # change after you push
!git clone -q {REPO_URL} derja-tts || echo "Clone failed: upload the repo folder to /content/derja-tts instead"
import sys
sys.path.insert(0, "derja-tts/src")

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

ckpt_dir = Path(snapshot_download("Ghazouaniwala/silma-tts-derja"))
print(sorted(str(p.relative_to(ckpt_dir)) for p in ckpt_dir.rglob("*") if p.is_file()))

def find(*patterns):
    for pat in patterns:
        hits = sorted(ckpt_dir.rglob(pat))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"none of {patterns} in the repo, check the listing above")

CKPT = find("model.pt", "*.safetensors", "*.pt")
VOCAB = find("vocab.txt")
CFG = find("config.yaml", "*.yaml")
print(CKPT, VOCAB, CFG, sep="\n")

In [ ]:
from google.colab import files
print("Upload 10-15 s of YOU speaking Derja (wav or mp3). Quiet room, no music, natural narration tone.")
uploaded = files.upload()
REF = next(iter(uploaded))
print("reference:", REF)

# F5-TTS needs the exact words of the reference clip (Arabic script is fine).
# Leave it empty and F5 will transcribe it with Whisper, which is bad at Derja.
REF_TEXT = ""

In [ ]:
from derja_tts.text import Normalizer
norm = Normalizer()

lines = [l.strip() for l in open("derja-tts/examples/sentences.txt", encoding="utf-8") if l.strip()]
# Add 2-3 lines copied from one of YOUR real video scripts, exactly as you wrote them:
lines += [
    # "...",
]

for i, line in enumerate(lines):
    r = norm.analyze(line)
    print(f"[{i:02d}] raw : {line}")
    print(f"     norm: {r.text}")
    if r.unknown:
        print(f"     unknown (kept Latin): {r.unknown}")

## Generate
If the CLI complains about a flag, run `!f5-tts_infer-cli --help`: flag names can change between F5-TTS versions.

In [ ]:
import os, subprocess
from IPython.display import Audio, display

OUT = "out_silma"
os.makedirs(OUT, exist_ok=True)

def say(text, name):
    cmd = ["f5-tts_infer-cli", "-mc", str(CFG), "-p", str(CKPT), "-v", str(VOCAB),
           "-r", REF, "-s", REF_TEXT, "-t", text, "-o", OUT, "-w", name]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(res.stderr[-2000:])
    return f"{OUT}/{name}"

for i, line in enumerate(lines):
    normalized = norm(line)
    variants = [("norm", normalized)] if normalized == line else [("raw", line), ("norm", normalized)]
    for tag, text in variants:
        print(f"[{i:02d}] {tag}: {text}")
        path = say(text, f"{i:02d}_{tag}.wav")
        if os.path.exists(path):
            display(Audio(path))

## Score it
Same sheet as 01a, so the two baselines compare line by line.

In [ ]:
# Listen to every pair, then fill this in. 1 = bad, 5 = perfect.
#   tunisian   : does the Derja sound Tunisian (not MSA, not Levantine)?
#   tech       : are the English/French terms pronounced right?
#   voice      : does it sound like you?
#   better     : "raw", "norm" or "same"
scores = {
    # 0: dict(tunisian=3, tech=4, voice=4, better="norm", notes="overfitting OK, barcha too MSA"),
}

import csv
with open(f"{OUT}/scores.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["line", "raw", "norm", "tunisian", "tech", "voice", "better", "notes"])
    for i, s in sorted(scores.items()):
        w.writerow([i, lines[i], norm(lines[i]), s.get("tunisian"), s.get("tech"),
                    s.get("voice"), s.get("better"), s.get("notes", "")])
print("saved", f"{OUT}/scores.csv")

In [ ]:
from google.colab import files
!zip -qr {OUT}.zip {OUT}
files.download(f"{OUT}.zip")